<a href="https://colab.research.google.com/github/rounak393/clab/blob/main/fft.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [1]:
import os
os.environ["KAGGLE_API_TOKEN"]='KGAT_c03d989b55c966d18c971a92b023645b'

In [2]:
!kaggle datasets download -d britikak/busi-dataset

Dataset URL: https://www.kaggle.com/datasets/britikak/busi-dataset
License(s): unknown
100% 195M/195M [00:03<00:00, 60.5MB/s]



In [3]:
!unzip -q busi-dataset.zip -d busi_dataset

In [4]:
!pip install albumentations -q

import os
import copy
import torch
import random
import numpy as np
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader, Subset
from PIL import Image
import albumentations as A
from tqdm import tqdm

torch.backends.cudnn.benchmark = True
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

BASE_DIR = "/content/busi_dataset/Dataset_BUSI_with_GT"
CLASSES = ["benign", "malignant"]
IMG_SIZE = 256
BATCH_SIZE = 4
EPOCHS = 50

train_transform = A.Compose([
    A.Resize(IMG_SIZE, IMG_SIZE),
    A.HorizontalFlip(p=0.5),
    A.VerticalFlip(p=0.5),
    A.ShiftScaleRotate(shift_limit=0.1, scale_limit=0.1, rotate_limit=15, p=0.5),
    A.Normalize(mean=(0.485, 0.456, 0.406), std=(0.229, 0.224, 0.225))
])

val_transform = A.Compose([
    A.Resize(IMG_SIZE, IMG_SIZE),
    A.Normalize(mean=(0.485, 0.456, 0.406), std=(0.229, 0.224, 0.225))
])

class BUSISegmentationDataset(Dataset):
    def __init__(self, base_dir, classes, transform=None):
        self.samples = []
        self.transform = transform
        for cls in classes:
            cls_dir = os.path.join(base_dir, cls)
            if not os.path.exists(cls_dir): continue
            images = [f for f in os.listdir(cls_dir) if f.endswith(".png") and "mask" not in f]
            for img_name in images:
                img_path = os.path.join(cls_dir, img_name)
                base_name = img_name.replace(".png", "")
                mask_files = [f for f in os.listdir(cls_dir) if f.startswith(base_name + "_mask")]
                if not mask_files: continue
                self.samples.append((img_path, [os.path.join(cls_dir, f) for f in mask_files]))

    def __len__(self): return len(self.samples)

    def __getitem__(self, idx):
        img_path, mask_paths = self.samples[idx]
        image = np.array(Image.open(img_path).convert("RGB"))
        combined_mask = np.zeros(image.shape[:2], dtype=np.uint8)
        for mpath in mask_paths:
            mask = np.array(Image.open(mpath).convert("L"))
            combined_mask = np.logical_or(combined_mask, (mask > 0).astype(np.uint8))
        combined_mask = combined_mask.astype(np.float32)

        if self.transform:
            augmented = self.transform(image=image, mask=combined_mask)
            image, combined_mask = augmented["image"], augmented["mask"]

        return torch.from_numpy(image).permute(2, 0, 1).float(), torch.from_numpy(combined_mask).unsqueeze(0).float()

full_dataset = BUSISegmentationDataset(BASE_DIR, classes=CLASSES, transform=None)
indices = list(range(len(full_dataset)))
np.random.shuffle(indices)

train_sz, val_sz = int(0.8 * len(full_dataset)), int(0.1 * len(full_dataset))
train_ds = Subset(BUSISegmentationDataset(BASE_DIR, CLASSES, train_transform), indices[val_sz:train_sz + val_sz])
val_ds = Subset(BUSISegmentationDataset(BASE_DIR, CLASSES, val_transform), indices[:val_sz])
test_ds = Subset(BUSISegmentationDataset(BASE_DIR, CLASSES, val_transform), indices[train_sz + val_sz:])

train_loader = DataLoader(train_ds, batch_size=BATCH_SIZE, shuffle=True, num_workers=2, pin_memory=True)
val_loader = DataLoader(val_ds, batch_size=BATCH_SIZE, shuffle=False, num_workers=2, pin_memory=True)
test_loader = DataLoader(test_ds, batch_size=BATCH_SIZE, shuffle=False, num_workers=2, pin_memory=True)

/usr/local/lib/python3.12/dist-packages/albumentations/core/validation.py:114: UserWarning: ShiftScaleRotate is a special case of Affine transform. Please use Affine transform instead.
  original_init(self, **validated_kwargs)


In [5]:
class ParameterFreeFFT(nn.Module):
    def __init__(self, mask_ratio=0.15):
        super().__init__()
        self.mask_ratio = mask_ratio

    def forward(self, x):
        _, _, H, W = x.shape
        fft_x = torch.fft.fftshift(torch.fft.fft2(x, norm="ortho"))

        mask = torch.ones_like(fft_x, device=x.device)
        center_h, center_w = H // 2, W // 2
        r_h, r_w = int(H * self.mask_ratio), int(W * self.mask_ratio)

        mask[:, :, center_h - r_h : center_h + r_h, center_w - r_w : center_w + r_w] = 0

        filtered_fft = fft_x * mask
        out = torch.fft.ifft2(torch.fft.ifftshift(filtered_fft), norm="ortho").real
        return out

class DoubleConv(nn.Module):
    def __init__(self, in_c, out_c):
        super().__init__()
        self.net = nn.Sequential(
            nn.Conv2d(in_c, out_c, 3, padding=1, bias=False),
            nn.BatchNorm2d(out_c),
            nn.ReLU(inplace=True),
            nn.Conv2d(out_c, out_c, 3, padding=1, bias=False),
            nn.BatchNorm2d(out_c),
            nn.ReLU(inplace=True)
        )
    def forward(self, x): return self.net(x)

In [6]:
class FFTUNet(nn.Module):
    def __init__(self, out_channels=1):
        super().__init__()
        self.fft_filter = ParameterFreeFFT()

        self.enc1 = DoubleConv(3, 64)
        self.enc2 = DoubleConv(64, 128)
        self.enc3 = DoubleConv(128, 256)
        self.enc4 = DoubleConv(256, 512)
        self.bottleneck = DoubleConv(512, 1024)

        self.pool = nn.MaxPool2d(2)

        self.up4 = nn.ConvTranspose2d(1024, 512, 2, stride=2)
        self.dec4 = DoubleConv(1024 + 512, 512)

        self.up3 = nn.ConvTranspose2d(512, 256, 2, stride=2)
        self.dec3 = DoubleConv(512 + 256, 256)

        self.up2 = nn.ConvTranspose2d(256, 128, 2, stride=2)
        self.dec2 = DoubleConv(256 + 128, 128)

        self.up1 = nn.ConvTranspose2d(128, 64, 2, stride=2)
        self.dec1 = DoubleConv(128 + 64, 64)

        self.final_conv = nn.Conv2d(64, out_channels, kernel_size=1)

    def forward(self, x):
        e1 = self.enc1(x)
        e2 = self.enc2(self.pool(e1))
        e3 = self.enc3(self.pool(e2))
        e4 = self.enc4(self.pool(e3))
        b = self.bottleneck(self.pool(e4))

        f4 = torch.cat([e4, self.fft_filter(e4)], dim=1)
        d4 = self.dec4(torch.cat([self.up4(b), f4], dim=1))

        f3 = torch.cat([e3, self.fft_filter(e3)], dim=1)
        d3 = self.dec3(torch.cat([self.up3(d4), f3], dim=1))

        f2 = torch.cat([e2, self.fft_filter(e2)], dim=1)
        d2 = self.dec2(torch.cat([self.up2(d3), f2], dim=1))

        f1 = torch.cat([e1, self.fft_filter(e1)], dim=1)
        d1 = self.dec1(torch.cat([self.up1(d2), f1], dim=1))

        return self.final_conv(d1)

In [7]:
class StrictBCEDiceLoss(nn.Module):
    def __init__(self, smooth=1e-5):
        super().__init__()
        self.smooth = smooth
        self.bce = nn.BCEWithLogitsLoss()
    def forward(self, logits, targets):
        bce_loss = self.bce(logits, targets)
        preds = torch.sigmoid(logits).view(-1)
        targets_f = targets.view(-1)
        inter = (preds * targets_f).sum()
        dice_loss = 1 - (2 * inter + self.smooth) / (preds.sum() + targets_f.sum() + self.smooth)
        return 0.2 * bce_loss + 0.8 * dice_loss

def strict_dice_coef(y_true, logits, smooth=1e-5):
    y_pred = (torch.sigmoid(logits) > 0.5).float().view(-1)
    y_true_f = y_true.view(-1)
    inter = (y_true_f * y_pred).sum()
    return (2. * inter + smooth) / (y_true_f.sum() + y_pred.sum() + smooth)

In [8]:
model = FFTUNet(out_channels=1).to(device)
criterion = StrictBCEDiceLoss()
optimizer = torch.optim.Adam(model.parameters(), lr=2e-4, weight_decay=1e-4)
scheduler = torch.optim.lr_scheduler.CosineAnnealingLR(optimizer, T_max=EPOCHS)

best_val_dice = 0.0
best_model_weights = None
best_epoch = 0

print("")

for epoch in range(EPOCHS):
    model.train()
    train_loss = train_dice = 0

    for images, masks in tqdm(train_loader, desc=f"Epoch {epoch+1}/{EPOCHS}"):
        images, masks = images.to(device, non_blocking=True), masks.to(device, non_blocking=True)
        optimizer.zero_grad(set_to_none=True)

        logits = model(images)
        loss = criterion(logits, masks)

        loss.backward()
        optimizer.step()

        train_loss += loss.item()
        with torch.no_grad():
            train_dice += strict_dice_coef(masks, logits).item()

    scheduler.step()

    model.eval()
    val_loss = val_dice = 0
    with torch.no_grad():
        for images, masks in val_loader:
            images, masks = images.to(device, non_blocking=True), masks.to(device, non_blocking=True)

            logits = model(images)
            loss = criterion(logits, masks)

            val_loss += loss.item()
            val_dice += strict_dice_coef(masks, logits).item()

    avg_t_loss = train_loss / len(train_loader)
    avg_t_dice = train_dice / len(train_loader)
    avg_v_loss = val_loss / len(val_loader)
    avg_v_dice = val_dice / len(val_loader)

    print(f"Epoch [{epoch+1}/{EPOCHS}] | Train Dice: {avg_t_dice:.4f} | Train Loss: {avg_t_loss:.4f} | Val Dice: {avg_v_dice:.4f} | Val Loss: {avg_v_loss:.4f}")

    if avg_v_dice > best_val_dice:
        best_val_dice = avg_v_dice
        best_epoch = epoch + 1
        best_model_weights = copy.deepcopy(model.state_dict())

print("\n==========================================================")
print(f" BEST MODEL FOUND AT EPOCH: {best_epoch} (Val Dice: {best_val_dice:.4f}) ")
print("==========================================================")

model.load_state_dict(best_model_weights)
model.eval()

test_loss = test_dice = 0
with torch.no_grad():
    for images, masks in test_loader:
        images, masks = images.to(device, non_blocking=True), masks.to(device, non_blocking=True)

        logits = model(images)
        loss = criterion(logits, masks)

        test_loss += loss.item()
        test_dice += strict_dice_coef(masks, logits).item()

avg_test_loss = test_loss / len(test_loader)
avg_test_dice = test_dice / len(test_loader)

print(f"\n FINAL TEST DICE SCORE: {avg_test_dice:.4f}")
print(f" FINAL TEST LOSS: {avg_test_loss:.4f}")

Epoch 1/50: 100%|██████████| 130/130 [01:07<00:00,  1.92it/s]


Epoch [1/50] | Train Dice: 0.3453 | Train Loss: 0.7259 | Val Dice: 0.4761 | Val Loss: 0.6522


Epoch 2/50: 100%|██████████| 130/130 [00:49<00:00,  2.62it/s]


Epoch [2/50] | Train Dice: 0.4757 | Train Loss: 0.6046 | Val Dice: 0.4899 | Val Loss: 0.5997


Epoch 3/50: 100%|██████████| 130/130 [00:48<00:00,  2.69it/s]


...

Epoch [48/50] | Train Dice: 0.7775 | Train Loss: 0.2233 | Val Dice: 0.7372 | Val Loss: 0.2602


Epoch 49/50: 100%|██████████| 130/130 [00:49<00:00,  2.65it/s]


Epoch [49/50] | Train Dice: 0.7876 | Train Loss: 0.2142 | Val Dice: 0.7404 | Val Loss: 0.2565


Epoch 50/50: 100%|██████████| 130/130 [00:49<00:00,  2.65it/s]


Epoch [50/50] | Train Dice: 0.7716 | Train Loss: 0.2298 | Val Dice: 0.7328 | Val Loss: 0.2642

 BEST MODEL FOUND AT EPOCH: 35 (Val Dice: 0.7468) 

 FINAL TEST DICE SCORE: 0.7004
 FINAL TEST LOSS: 0.2850
